# CCE Degradation Under Proton Irradiation

This notebook predicts charge collection efficiency (CCE) degradation in a 10 um 4H-SiC
detector under 62 MeV proton irradiation across a fluence range of $10^{10}$ to $10^{15}$ cm$^{-2}$.

The simulation couples the radiation damage physics model (defect introduction, carrier lifetime
degradation, carrier removal) from Burin et al. (arXiv:2407.16710, 2024) to a 1D drift-diffusion
solver (devsim) using the fluence-as-temperature architecture pattern: a fresh device is created
for each fluence point with damaged parameters applied before Poisson equilibrium.

**Sections:**
1. CCE vs fluence at reference bias ($V = -40$ V)
2. CCE vs fluence at multiple bias voltages
3. CCE vs bias at fixed fluence levels
4. Sensitivity analysis: linear vs logarithmic lifetime models with uncertainty bands
5. Summary table and discussion

**Reference:** Burin et al., arXiv:2407.16710 (2024) -- damage constants for 4H-SiC

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.charge_collection import cce_vs_fluence, cce_vs_bias_at_fluence
from src.radiation_damage import RadiationDamageParams

# Publication-quality settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'figure.figsize': (8, 6),
})

print('Notebook 10: CCE Degradation Under Proton Irradiation')
print('Device: 10 um 4H-SiC epitaxial detector')
print('Beam: 62 MeV protons')

In [ ]:
# Define fluence grid: 12 points in logspace from 1e10 to 1e15 cm^-2
# - 1e10: negligible damage (near-pristine)
# - 1e15: well beyond full compensation of bulk doping
# Note: solver may return NaN above ~5e13 p/cm^2 for 62 MeV protons
# due to full doping compensation (N_D_bulk = 8.5e13 cm^-3)
fluences = np.geomspace(1e10, 1e15, 12)
print(f'Fluence grid: {len(fluences)} points from {fluences[0]:.0e} to {fluences[-1]:.0e} cm^-2')
for i, phi in enumerate(fluences):
    print(f'  [{i:2d}] {phi:.2e} cm^-2')

In [ ]:
# Figure 1: CCE vs fluence at reference bias (-40V)
result_ref = cce_vs_fluence(fluences, V_bias=-40.0, lifetime_model='linear')

fig, ax = plt.subplots(figsize=(8, 6))
ax.semilogx(result_ref['fluences'], result_ref['cce_values'], 'o-',
            color='#1f77b4', markersize=6, linewidth=2,
            label='Linear model, $V = -40$ V')

ax.set_xlabel('Proton fluence (cm$^{-2}$)')
ax.set_ylabel('CCE')
ax.set_title('CCE vs Proton Fluence (62 MeV, $V = -40$ V)')
ax.set_ylim(0, 1.05)
ax.set_xlim(1e10, 1e15)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('../figures/10_cce_vs_fluence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/10_cce_vs_fluence.png')

In [ ]:
# Figure 2: CCE vs fluence at multiple bias voltages (CCED-02)
# Higher bias extends the depletion region and partially compensates damage
bias_voltages = [-20, -40, -60, -80, -100]
colors = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

results_multibias = {}
for V in bias_voltages:
    print(f'Computing CCE vs fluence at V = {V} V ...')
    results_multibias[V] = cce_vs_fluence(fluences, V_bias=float(V), lifetime_model='linear')

fig, ax = plt.subplots(figsize=(8, 6))
for V, color in zip(bias_voltages, colors):
    r = results_multibias[V]
    ax.semilogx(r['fluences'], r['cce_values'], 'o-',
                color=color, markersize=5, linewidth=1.8,
                label=f'$V = {V}$ V')

ax.set_xlabel('Proton fluence (cm$^{-2}$)')
ax.set_ylabel('CCE')
ax.set_title('CCE vs Proton Fluence at Multiple Bias Voltages')
ax.set_ylim(0, 1.05)
ax.set_xlim(1e10, 1e15)
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('../figures/10_cce_vs_fluence_multibias.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/10_cce_vs_fluence_multibias.png')

In [ ]:
# Figure 3: CCE vs bias at fixed fluence levels (CCED-03)
# Shows CCE recovery with increasing reverse bias at each damage level
# Note: max fluence limited to 5e13 p/cm^2 to stay within solver convergence
# regime (N_D_bulk = 8.5e13 cm^-3; full compensation at ~5e13 p/cm^2 for 62 MeV)
fluence_levels = [0, 1e12, 1e13, 5e13]
fluence_labels = ['Pristine', '$10^{12}$ p/cm$^2$', '$10^{13}$ p/cm$^2$', '$5{\\times}10^{13}$ p/cm$^2$']
colors_bias = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
V_range = np.arange(-10, -110, -10, dtype=float)

results_bias = {}
for phi, label in zip(fluence_levels, fluence_labels):
    print(f'Computing CCE vs bias at fluence = {phi:.0e} p/cm^2 ...')
    try:
        results_bias[phi] = cce_vs_bias_at_fluence(
            V_range, fluence=float(phi), lifetime_model='linear'
        )
    except Exception as e:
        print(f'  Solver diverged at fluence={phi:.0e}: {e}')
        results_bias[phi] = None

fig, ax = plt.subplots(figsize=(8, 6))
for (phi, label), color in zip(zip(fluence_levels, fluence_labels), colors_bias):
    r = results_bias[phi]
    if r is not None:
        ax.plot(np.abs(r['voltages']), r['cce_values'], 'o-',
                color=color, markersize=5, linewidth=1.8,
                label=label)

ax.set_xlabel('Reverse bias $|V|$ (V)')
ax.set_ylabel('CCE')
ax.set_title('CCE vs Bias Voltage at Fixed Fluence Levels')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('../figures/10_cce_vs_bias_damaged.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/10_cce_vs_bias_damaged.png')

In [ ]:
# Figure 4: Sensitivity analysis -- linear vs logarithmic models with uncertainty bands (NBKV-02)
# Two-panel figure: left = linear, right = logarithmic
# Uncertainty bands from scaling all eta values by 0.5x and 2.0x
fluences_sens = np.geomspace(1e10, 1e15, 8)  # 8 points for faster runtime

# Scale damage constants
def scaled_damage_params(scale):
    """Create RadiationDamageParams with all eta values scaled."""
    p = RadiationDamageParams()
    p.eta_Z12 *= scale
    p.eta_EH67 *= scale
    p.eta_EH4 *= scale
    p.eta_removal *= scale
    return p

scales = [0.5, 1.0, 2.0]
scale_labels = ['0.5x', '1.0x (nominal)', '2.0x']

sensitivity_results = {}
for model in ['linear', 'logarithmic']:
    sensitivity_results[model] = {}
    for scale in scales:
        dp = scaled_damage_params(scale) if scale != 1.0 else None
        print(f'Computing: {model} model, {scale}x damage constants ...')
        sensitivity_results[model][scale] = cce_vs_fluence(
            fluences_sens, V_bias=-40.0, lifetime_model=model, damage_params=dp
        )

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, model, title in [(ax1, 'linear', 'Linear Lifetime Model'),
                          (ax2, 'logarithmic', 'Logarithmic Lifetime Model')]:
    r_nom = sensitivity_results[model][1.0]
    r_lo = sensitivity_results[model][0.5]
    r_hi = sensitivity_results[model][2.0]

    # Uncertainty band
    ax.fill_between(r_nom['fluences'], r_lo['cce_values'], r_hi['cce_values'],
                    alpha=0.3, color='#1f77b4', label='0.5x--2.0x envelope')
    # Nominal curve
    ax.semilogx(r_nom['fluences'], r_nom['cce_values'], 'o-',
                color='#1f77b4', markersize=5, linewidth=2,
                label='Nominal (1.0x)')

    ax.set_xlabel('Proton fluence (cm$^{-2}$)')
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.set_xlim(1e10, 1e15)
    ax.legend(loc='lower left')
    ax.grid(True, alpha=0.3)

ax1.set_ylabel('CCE')
fig.suptitle('CCE Sensitivity to Damage Constants ($V = -40$ V, 62 MeV protons)', fontsize=14)
fig.tight_layout()
fig.savefig('../figures/10_cce_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/10_cce_sensitivity.png')

In [ ]:
# Summary table: CCE at key fluence milestones for -40V bias
# Compare linear and logarithmic lifetime models
milestone_fluences = np.array([1e11, 1e12, 1e13, 1e14])

print('Computing CCE at milestone fluences (linear model) ...')
r_lin = cce_vs_fluence(milestone_fluences, V_bias=-40.0, lifetime_model='linear')
print('Computing CCE at milestone fluences (logarithmic model) ...')
r_log = cce_vs_fluence(milestone_fluences, V_bias=-40.0, lifetime_model='logarithmic')

print()
print('=' * 65)
print('CCE at Key Fluence Milestones (V = -40 V, 62 MeV protons)')
print('=' * 65)
print(f'{"Fluence (p/cm2)":>18s}  {"CCE (linear)":>14s}  {"CCE (logarithmic)":>18s}')
print('-' * 65)
for i, phi in enumerate(milestone_fluences):
    cce_l = r_lin['cce_values'][i]
    cce_g = r_log['cce_values'][i]
    cce_l_str = f'{cce_l:.4f}' if not np.isnan(cce_l) else 'NaN (diverged)'
    cce_g_str = f'{cce_g:.4f}' if not np.isnan(cce_g) else 'NaN (diverged)'
    print(f'{phi:>18.0e}  {cce_l_str:>14s}  {cce_g_str:>18s}')
print('=' * 65)
print()
print('Note: NaN indicates Newton solver divergence near full doping compensation.')

## Discussion

### Key Observations

**CCE degradation thresholds** (at $V = -40$ V, linear model):
- CCE remains above 80% for fluences below approximately $10^{12}$ p/cm$^2$
- CCE drops below 50% in the range $10^{12}$--$10^{13}$ p/cm$^2$ depending on model
- Above $\sim 5 \times 10^{13}$ p/cm$^2$, the bulk doping ($N_{D,\mathrm{bulk}} = 8.5 \times 10^{13}$ cm$^{-3}$)
  is fully compensated by carrier removal ($\eta_{\mathrm{removal}} = 5.0$ cm$^{-1}$),
  and the Newton solver diverges

### Effect of Bias Voltage

Higher reverse bias partially compensates radiation damage by extending the depletion region
into the damaged epitaxial layer. At $V = -100$ V, the detector maintains useful CCE at
fluences where $V = -20$ V has already degraded significantly. This implies that
operating at higher bias is a practical radiation-hardness strategy.

### Linear vs Logarithmic Lifetime Models

The two lifetime models agree at low fluence ($< 10^{12}$ p/cm$^2$) but diverge at higher
damage levels:
- The **linear model** ($1/\tau = 1/\tau_0 + K_\tau \Phi$) predicts more rapid degradation
- The **logarithmic model** ($\tau = \tau_0 / (1 + K_\tau \tau_0 \Phi)^\alpha$, $\alpha = 0.8$)
  predicts more gradual saturation due to effective trap filling at high defect concentrations

### Uncertainty Bands

The 0.5x--2.0x damage constant envelope provides a practical estimate of prediction
uncertainty given the scatter in measured introduction rates. The uncertainty grows
with fluence, as expected. For the logarithmic model, the uncertainty band is narrower
at high fluence due to the sub-linear damage accumulation.

### Limitations

- NIEL hardness factor for 62 MeV protons ($\kappa = 0.35$) is interpolated; validation
  against SR-NIEL calculator output is pending
- Solver diverges near full doping compensation ($\Phi > \Phi_{\mathrm{crit}}$);
  explicit handling deferred to Phase 16
- Uniform doping approximation in the bulk; graded epi profile effects on radiation
  tolerance are a future study